# Sentence Corpus Inspection

In [1]:
import os
import pandas as pd

PATH = "/home/tom/data/sentence_corpus.csv"

size_gb = os.path.getsize(PATH) / 1e9
print(f"File size: {size_gb:.2f} GB")

File size: 20.67 GB


In [2]:
# Actual row count — reads only the 'country' column (fast, handles quoted newlines)
total = 0
for chunk in pd.read_csv(PATH, usecols=["country"], chunksize=1_000_000):
    total += len(chunk)
print(f"Actual row count: {total:,}")

Actual row count: 56,164,815


In [3]:
# Also count raw newlines so we can see the discrepancy
with open(PATH, "rb") as f:
    raw_lines = sum(1 for _ in f) - 1
print(f"Raw newline count (overcounts quoted fields): {raw_lines:,}")
print(f"Difference (embedded newlines in sentences):  {raw_lines - total:,}")

Raw newline count (overcounts quoted fields): 1,395,741,905
Difference (embedded newlines in sentences):  1,339,577,090


In [4]:
# Load a sample to inspect structure
df = pd.read_csv(PATH, nrows=100_000)
print(df.dtypes)
df.head()

,sentence,sentence_idx,date,speaker,country,source_file,source_speech_id,source_dataset,source_dataset_type
0,Please \n take \n a \n seat \n .,0,1996-01-15,PAD_00334,AT,ParlaMint-AT-en_1996-01-15-020-XX-NRSITZ-00001...,ParlaMint-AT_1996-01-15-020-XX-NRSITZ-00001_d7...,ParlaMint-AT,parlamint
1,– \n I \n also \n ask \n the \n photographers ...,1,1996-01-15,PAD_00334,AT,ParlaMint-AT-en_1996-01-15-020-XX-NRSITZ-00001...,ParlaMint-AT_1996-01-15-020-XX-NRSITZ-00001_d7...,ParlaMint-AT,parlamint
2,I \n may \n attend \n the \n first \n and \n c...,2,1996-01-15,PAD_00334,AT,ParlaMint-AT-en_1996-01-15-020-XX-NRSITZ-00001...,ParlaMint-AT_1996-01-15-020-XX-NRSITZ-00001_d7...,ParlaMint-AT,parlamint
3,The \n \n Council \n of \...,3,1996-01-15,PAD_00334,AT,ParlaMint-AT-en_1996-01-15-020-XX-NRSITZ-00001...,ParlaMint-AT_1996-01-15-020-XX-NRSITZ-00001_d7...,ParlaMint-AT,parlamint
4,"With \n respect \n , \n I \n welcome \n the \n...",4,1996-01-15,PAD_00334,AT,ParlaMint-AT-en_1996-01-15-020-XX-NRSITZ-00001...,ParlaMint-AT_1996-01-15-020-XX-NRSITZ-00001_d7...,ParlaMint-AT,parlamint


In [5]:
# Row counts by source_dataset_type
type_counts = {}
dataset_counts = {}
country_counts = {}

for chunk in pd.read_csv(PATH, usecols=["source_dataset_type", "source_dataset", "country"], chunksize=1_000_000):
    for col, acc in [("source_dataset_type", type_counts),
                     ("source_dataset",      dataset_counts),
                     ("country",             country_counts)]:
        counts = chunk[col].value_counts()
        for k, v in counts.items():
            acc[k] = acc.get(k, 0) + v

print("=== By source_dataset_type ===")
for k, v in sorted(type_counts.items(), key=lambda x: -x[1]):
    print(f"  {k:<25} {v:>15,}")

print("\n=== By source_dataset ===")
for k, v in sorted(dataset_counts.items(), key=lambda x: -x[1]):
    print(f"  {k:<30} {v:>15,}")

print("\n=== By country ===")
for k, v in sorted(country_counts.items(), key=lambda x: -x[1]):
    print(f"  {k:<10} {v:>15,}")

=== By source_dataset_type ===
  parlamint                      56,164,815

=== By source_dataset ===
  ParlaMint-GB                         4,994,352
  ParlaMint-NO                         4,649,594
  ParlaMint-NL                         4,589,138
  ParlaMint-HR                         4,468,902
  ParlaMint-AT                         4,448,320
  ParlaMint-PL                         3,137,374
  ParlaMint-ES                         2,990,949
  ParlaMint-GR                         2,867,231
  ParlaMint-BE                         2,417,494
  ParlaMint-FR                         2,394,539
  ParlaMint-DK                         2,200,154
  ParlaMint-CZ                         2,110,562
  ParlaMint-RS                         2,048,627
  ParlaMint-IS                         1,840,114
  ParlaMint-EE                         1,807,129
  ParlaMint-BG                         1,794,863
  ParlaMint-HU                         1,786,887
  ParlaMint-IT                         1,463,643
  ParlaMint-PT  

In [6]:
# Date range per source_dataset (sample-based — reads first 1M rows)
df_sample = pd.read_csv(PATH, usecols=["source_dataset", "date"], nrows=1_000_000)
df_sample["date"] = pd.to_datetime(df_sample["date"], errors="coerce")
df_sample.groupby("source_dataset")["date"].agg(["min", "max", "count"])

,min,max,count
source_dataset,,,
ParlaMint-AT,1996-01-15,2000-11-23,1000000


In [7]:
# Sentence length distribution (sample)
df_sample2 = pd.read_csv(PATH, usecols=["sentence", "source_dataset_type"], nrows=500_000)
df_sample2["len"] = df_sample2["sentence"].str.len()
print(df_sample2.groupby("source_dataset_type")["len"].describe().round(1))

                        count   mean    std  min   25%    50%    75%      max
source_dataset_type                                                          
parlamint            500000.0  188.6  212.6  1.0  46.0  132.0  271.0  10433.0


In [8]:
# Check for embedded newlines in sentences (what was overcounting the rows)
has_newline = df_sample2["sentence"].str.contains(r"\n", na=False)
print(f"Sentences with embedded newlines: {has_newline.sum():,} / {len(df_sample2):,} ({has_newline.mean()*100:.2f}%)")

Sentences with embedded newlines: 416,995 / 500,000 (83.40%)


In [9]:
# Inspect a few random rows
df.sample(10)[["sentence", "date", "speaker", "country", "source_dataset", "source_dataset_type"]]

,sentence,date,speaker,country,source_dataset,source_dataset_type
92136,Perhaps \n the \n rulers \n from \n the \n ...,1996-05-22,PAD_00074,AT,ParlaMint-AT,parlamint
92826,Scheibner: Have you already apologised for the...,1996-05-22,PAD_01902,AT,ParlaMint-AT,parlamint
32903,My \n \n Dear \n Ones \n ...,1996-03-20,PAD_01158,AT,ParlaMint-AT,parlamint
97306,It \n has \n already \n been \n done \n that \...,1996-05-23,PAD_01781,AT,ParlaMint-AT,parlamint
57017,The \n fact \n that \n we \n are \n continuing...,1996-04-16,PAD_02789,AT,ParlaMint-AT,parlamint
40200,I \n would \n particularly \n like \n to \n me...,1996-04-16,PAD_00334,AT,ParlaMint-AT,parlamint
36917,I \n can \n \n assure \n ...,1996-03-21,PAD_00125,AT,ParlaMint-AT,parlamint
49580,On \n the \n other \n hand \n ...,1996-04-16,PAD_01763,AT,ParlaMint-AT,parlamint
47582,"In \n any \n case \n \n ,...",1996-04-16,PAD_00401,AT,ParlaMint-AT,parlamint
58476,5 \n LEG \n on \n the \n motions \n for \n res...,1996-04-16,PAD_00334,AT,ParlaMint-AT,parlamint
